In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/orvile/wesad-wearable-stress-affect-detection-dataset/WESAD/wesad_readme.pdf
/kaggle/input/datasets/orvile/wesad-wearable-stress-affect-detection-dataset/WESAD/S14/S14.pkl
/kaggle/input/datasets/orvile/wesad-wearable-stress-affect-detection-dataset/WESAD/S14/S14_quest.csv
/kaggle/input/datasets/orvile/wesad-wearable-stress-affect-detection-dataset/WESAD/S14/S14_respiban.txt
/kaggle/input/datasets/orvile/wesad-wearable-stress-affect-detection-dataset/WESAD/S14/S14_readme.txt
/kaggle/input/datasets/orvile/wesad-wearable-stress-affect-detection-dataset/WESAD/S14/S14_E4_Data/HR.csv
/kaggle/input/datasets/orvile/wesad-wearable-stress-affect-detection-dataset/WESAD/S14/S14_E4_Data/IBI.csv
/kaggle/input/datasets/orvile/wesad-wearable-stress-affect-detection-dataset/WESAD/S14/S14_E4_Data/BVP.csv
/kaggle/input/datasets/orvile/wesad-wearable-stress-affect-detection-dataset/WESAD/S14/S14_E4_Data/EDA.csv
/kaggle/input/datasets/orvile/wesad-wearable-stress-affect-detection-da

In [2]:
import os
print(os.listdir("/kaggle/input"))

['datasets']


In [3]:
# ============================================================
# WESAD Stress Detection - 10 Models
# Kaggle Corrected Version with Auto Path Detection
# ============================================================

import os
import pickle
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, cohen_kappa_score, roc_auc_score, recall_score, precision_score

from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier, BaggingClassifier
from sklearn.neighbors import KNeighborsClassifier
from xgboost import XGBClassifier

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, LSTM, Bidirectional, Conv1D, MaxPooling1D, Flatten
from tensorflow.keras.optimizers import Adam

# ============================================================
# 1. AUTO-DETECT WESAD DATASET PATH
# ============================================================

print("Folders inside /kaggle/input:")
print(os.listdir("/kaggle/input"))

possible_paths = []

for root, dirs, files in os.walk("/kaggle/input"):
    if "S2" in dirs:
        possible_paths.append(root)

if len(possible_paths) == 0:
    raise FileNotFoundError(
        "WESAD dataset not found. Make sure you clicked Add Data in Kaggle and attached the WESAD dataset."
    )

BASE_PATH = possible_paths[0]

print("Using BASE_PATH:", BASE_PATH)
print("Dataset folders:", os.listdir(BASE_PATH)[:10])

# ============================================================
# 2. LOAD AND PREPROCESS ECG DATA
# ============================================================

fs = 700
window_seconds = 10
window_size = fs * window_seconds

all_X = []
all_y = []

for subject in os.listdir(BASE_PATH):
    subject_path = os.path.join(BASE_PATH, subject)

    if not os.path.isdir(subject_path):
        continue

    pkl_file = os.path.join(subject_path, f"{subject}.pkl")

    if not os.path.exists(pkl_file):
        continue

    with open(pkl_file, "rb") as f:
        data = pickle.load(f, encoding="latin1")

    ecg = data["signal"]["chest"]["ECG"]
    labels = data["label"]

    # Use only baseline = 1 and stress = 2
    mask = np.isin(labels, [1, 2])

    ecg = ecg[mask].reshape(-1)
    labels = labels[mask]

    # Convert labels: baseline = 0, stress = 1
    labels = np.where(labels == 2, 1, 0)

    for start in range(0, len(ecg) - window_size, window_size):
        end = start + window_size

        window = ecg[start:end]
        label_window = labels[start:end]

        label = int(np.round(np.mean(label_window)))

        all_X.append(window)
        all_y.append(label)

X = np.array(all_X)
y = np.array(all_y)

print("Final X shape:", X.shape)
print("Final y shape:", y.shape)
print("Class distribution:", np.bincount(y))

# Normalize each window
X = (X - X.mean(axis=1, keepdims=True)) / (X.std(axis=1, keepdims=True) + 1e-8)

# ============================================================
# 3. FEATURE EXTRACTION FOR ML MODELS
# ============================================================

def extract_features(X):
    features = []

    for w in X:
        features.append([
            np.mean(w),
            np.std(w),
            np.min(w),
            np.max(w),
            np.median(w),
            np.percentile(w, 25),
            np.percentile(w, 75),
            np.sum(w ** 2) / len(w),
            np.sum(np.diff(np.sign(w)) != 0)
        ])

    return np.array(features)

X_features = extract_features(X)

# ============================================================
# 4. TRAIN / VALIDATION / TEST SPLIT
# ============================================================

X_train_f, X_temp_f, y_train, y_temp = train_test_split(
    X_features, y, test_size=0.30, random_state=42, stratify=y
)

X_val_f, X_test_f, y_val, y_test = train_test_split(
    X_temp_f, y_temp, test_size=0.50, random_state=42, stratify=y_temp
)

scaler = StandardScaler()
X_train_f = scaler.fit_transform(X_train_f)
X_val_f = scaler.transform(X_val_f)
X_test_f = scaler.transform(X_test_f)

# Deep learning split
X_train_s, X_temp_s, y_train_s, y_temp_s = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)

X_val_s, X_test_s, y_val_s, y_test_s = train_test_split(
    X_temp_s, y_temp_s, test_size=0.50, random_state=42, stratify=y_temp_s
)

# Downsample ECG from 7000 to 700 for faster training
X_train_s = X_train_s[:, ::10]
X_val_s = X_val_s[:, ::10]
X_test_s = X_test_s[:, ::10]

X_train_s = X_train_s[..., np.newaxis]
X_val_s = X_val_s[..., np.newaxis]
X_test_s = X_test_s[..., np.newaxis]

print("ML Train Shape:", X_train_f.shape)
print("DL Train Shape:", X_train_s.shape)

# ============================================================
# 5. EVALUATION FUNCTION
# ============================================================

results = []

def evaluate_model(model_name, train_acc, val_acc, y_true, y_pred, y_prob):
    test_acc = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred)
    kappa = cohen_kappa_score(y_true, y_pred)
    recall = recall_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred)
    auc = roc_auc_score(y_true, y_prob)

    results.append([
        model_name,
        round(train_acc, 4),
        round(val_acc, 4),
        round(test_acc, 4),
        round(f1, 4),
        round(kappa, 4),
        round(recall, 4),
        round(precision, 4),
        round(auc, 4)
    ])

# ============================================================
# 6. MACHINE LEARNING MODELS
# ============================================================

ml_models = {
    "SVM": SVC(kernel="rbf", probability=True, random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42),
    "AdaBoost": AdaBoostClassifier(n_estimators=100, random_state=42),
    "KNN": KNeighborsClassifier(n_neighbors=5),
    "XGBoost": XGBClassifier(
        n_estimators=100,
        learning_rate=0.05,
        max_depth=4,
        eval_metric="logloss",
        random_state=42
    ),
    "Bagging": BaggingClassifier(n_estimators=100, random_state=42)
}

for name, model in ml_models.items():
    print(f"\nTraining {name}...")

    model.fit(X_train_f, y_train)

    train_acc = model.score(X_train_f, y_train)
    val_acc = model.score(X_val_f, y_val)

    y_pred = model.predict(X_test_f)
    y_prob = model.predict_proba(X_test_f)[:, 1]

    evaluate_model(name, train_acc, val_acc, y_test, y_pred, y_prob)

# ============================================================
# 7. DEEP LEARNING MODELS
# ============================================================

def train_dl_model(model_name, model, epochs=10, batch_size=32):
    print(f"\nTraining {model_name}...")

    model.compile(
        optimizer=Adam(learning_rate=0.001),
        loss="binary_crossentropy",
        metrics=["accuracy"]
    )

    history = model.fit(
        X_train_s, y_train_s,
        validation_data=(X_val_s, y_val_s),
        epochs=epochs,
        batch_size=batch_size,
        verbose=1
    )

    train_acc = history.history["accuracy"][-1]
    val_acc = history.history["val_accuracy"][-1]

    y_prob = model.predict(X_test_s).reshape(-1)
    y_pred = (y_prob >= 0.5).astype(int)

    evaluate_model(model_name, train_acc, val_acc, y_test_s, y_pred, y_prob)

# ANN
ann_model = Sequential([
    Flatten(input_shape=(X_train_s.shape[1], 1)),
    Dense(128, activation="relu"),
    Dropout(0.3),
    Dense(64, activation="relu"),
    Dropout(0.3),
    Dense(1, activation="sigmoid")
])

train_dl_model("ANN", ann_model)

# LSTM
lstm_model = Sequential([
    LSTM(64, input_shape=(X_train_s.shape[1], 1)),
    Dropout(0.3),
    Dense(64, activation="relu"),
    Dense(1, activation="sigmoid")
])

train_dl_model("LSTM", lstm_model)

# Bi-LSTM
bilstm_model = Sequential([
    Bidirectional(LSTM(64), input_shape=(X_train_s.shape[1], 1)),
    Dropout(0.3),
    Dense(64, activation="relu"),
    Dense(1, activation="sigmoid")
])

train_dl_model("Bi-LSTM", bilstm_model)

# 1D-CNN
cnn_model = Sequential([
    Conv1D(32, kernel_size=3, activation="relu", input_shape=(X_train_s.shape[1], 1)),
    MaxPooling1D(pool_size=2),
    Conv1D(64, kernel_size=3, activation="relu"),
    MaxPooling1D(pool_size=2),
    Conv1D(128, kernel_size=3, activation="relu"),
    MaxPooling1D(pool_size=2),
    Flatten(),
    Dense(128, activation="relu"),
    Dropout(0.3),
    Dense(1, activation="sigmoid")
])

train_dl_model("1D-CNN", cnn_model)

# ============================================================
# 8. FINAL RESULTS TABLE
# ============================================================

results_df = pd.DataFrame(results, columns=[
    "Model",
    "Train Accuracy",
    "Validation Accuracy",
    "Test Accuracy",
    "F1 Score",
    "Kappa",
    "Recall",
    "Precision",
    "ROC-AUC"
])

print("\n================ FINAL RESULTS ================")
display(results_df)

results_df.to_csv("wesad_10_model_results.csv", index=False)

print("Results saved as wesad_10_model_results.csv")

2026-05-03 15:37:04.414406: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1777822624.663096      16 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777822624.737925      16 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1777822625.336363      16 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777822625.336414      16 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777822625.336417      16 computation_placer.cc:177] computation placer alr

Folders inside /kaggle/input:
['datasets']
Using BASE_PATH: /kaggle/input/datasets/orvile/wesad-wearable-stress-affect-detection-dataset/WESAD
Dataset folders: ['S14', 'S11', 'S13', 'S10', 'S8', 'S5', 'S7', 'S9', 'S15', 'wesad_readme.pdf']
Final X shape: (2749, 7000)
Final y shape: (2749,)
Class distribution: [1762  987]
ML Train Shape: (1924, 9)
DL Train Shape: (1924, 700, 1)

Training SVM...

Training Random Forest...

Training AdaBoost...

Training KNN...

Training XGBoost...

Training Bagging...

Training ANN...
Epoch 1/10


2026-05-03 15:39:42.647306: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


61/61 ━━━━━━━━━━━━━━━━━━━━ 2s 9ms/step - accuracy: 0.5749 - loss: 0.8186 - val_accuracy: 0.6432 - val_loss: 0.6590
Epoch 2/10
61/61 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.6647 - loss: 0.6227 - val_accuracy: 0.6383 - val_loss: 0.6557
Epoch 3/10
61/61 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7157 - loss: 0.5515 - val_accuracy: 0.6189 - val_loss: 0.6653
Epoch 4/10
61/61 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.7931 - loss: 0.4411 - val_accuracy: 0.6286 - val_loss: 0.7068
Epoch 5/10
61/61 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.8585 - loss: 0.3648 - val_accuracy: 0.6238 - val_loss: 0.7750
Epoch 6/10
61/61 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.9127 - loss: 0.2585 - val_accuracy: 0.6068 - val_loss: 0.8514
Epoch 7/10
61/61 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.9191 - loss: 0.2308 - val_accuracy: 0.6044 - val_loss: 0.9194
Epoch 8/10
61/61 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.9386 - loss: 0.1849 - val_accuracy: 0.6068 - val_loss: 1.0053
Epo

,Model,Train Accuracy,Validation Accuracy,Test Accuracy,F1 Score,Kappa,Recall,Precision,ROC-AUC
0,SVM,0.9381,0.9199,0.9395,0.9141,0.8674,0.8986,0.9301,0.9748
1,Random Forest,1.0000,0.9369,0.9637,0.9481,0.9202,0.9257,0.9716,0.9925
2,AdaBoost,0.9080,0.8689,0.9128,0.8750,0.8082,0.8514,0.9000,0.9755
3,KNN,0.9605,0.9272,0.9322,0.9048,0.8521,0.8986,0.9110,0.9820
4,XGBoost,0.9522,0.9175,0.9492,0.9283,0.8889,0.9189,0.9379,0.9829
5,Bagging,1.0000,0.9345,0.9540,0.9343,0.8989,0.9122,0.9574,0.9861
6,ANN,0.9475,0.5995,0.6053,0.3707,0.0916,0.3243,0.4324,0.5479
7,LSTM,0.6861,0.6286,0.6416,0.0133,0.0038,0.0068,0.5000,0.7713
8,Bi-LSTM,0.7110,0.7840,0.7191,0.5704,0.3646,0.5203,0.6311,0.7666
9,1D-CNN,0.9844,0.8689,0.8862,0.8498,0.7587,0.8986,0.8061,0.9601


Results saved as wesad_10_model_results.csv


In [4]:
# ============================================================
# WESAD Stress Detection - Method 1: xLSTM-style LSTM
#  Dataset: WESAD ECG, Binary Classification: Baseline vs Stress
# ============================================================

import os
import pickle
import numpy as np
import warnings
warnings.filterwarnings("ignore")

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, cohen_kappa_score, roc_auc_score

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

# ============================================================
# 1. Auto-detect WESAD dataset path in Kaggle
# ============================================================

possible_paths = []

for root, dirs, files in os.walk("/kaggle/input"):
    if "S2" in dirs:
        possible_paths.append(root)

if len(possible_paths) == 0:
    raise FileNotFoundError(
        "WESAD dataset not found. Please click Add Data in Kaggle and attach the WESAD dataset."
    )

BASE_PATH = possible_paths[0]

print("Using dataset path:", BASE_PATH)
print("Dataset folders:", os.listdir(BASE_PATH)[:10])

# ============================================================
# 2. Load ECG data from WESAD
# ============================================================

all_X = []
all_y = []

fs = 700
window_seconds = 10
window_size = fs * window_seconds

subjects = os.listdir(BASE_PATH)

for subject in subjects:
    subject_path = os.path.join(BASE_PATH, subject)

    if not os.path.isdir(subject_path):
        continue

    file_path = os.path.join(subject_path, f"{subject}.pkl")

    if not os.path.exists(file_path):
        continue

    with open(file_path, "rb") as f:
        data = pickle.load(f, encoding="latin1")

    ecg = data["signal"]["chest"]["ECG"]
    labels = data["label"]

    # Keep only baseline = 1 and stress = 2
    mask = np.isin(labels, [1, 2])

    ecg = ecg[mask].reshape(-1)
    labels = labels[mask]

    # Convert labels: baseline = 0, stress = 1
    labels = np.where(labels == 2, 1, 0)

    for start in range(0, len(ecg) - window_size, window_size):
        end = start + window_size

        window = ecg[start:end]
        label_window = labels[start:end]

        label = int(np.round(np.mean(label_window)))

        all_X.append(window)
        all_y.append(label)

X = np.array(all_X)
y = np.array(all_y)

print("Final X shape:", X.shape)
print("Final y shape:", y.shape)
print("Class distribution:", np.bincount(y))

# ============================================================
# 3. Normalize ECG windows
# ============================================================

X = (X - X.mean(axis=1, keepdims=True)) / (X.std(axis=1, keepdims=True) + 1e-8)

# ============================================================
# 4. Train / validation / test split
# ============================================================

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp
)

# Add channel dimension: (samples, 7000, 1)
X_train = X_train[..., np.newaxis]
X_val = X_val[..., np.newaxis]
X_test = X_test[..., np.newaxis]

print("Train:", X_train.shape)
print("Validation:", X_val.shape)
print("Test:", X_test.shape)

# ============================================================
# 5. Downsample for faster training
# ============================================================

X_train_ds = X_train[:, ::10, :]
X_val_ds = X_val[:, ::10, :]
X_test_ds = X_test[:, ::10, :]

print("Downsampled Train:", X_train_ds.shape)
print("Downsampled Validation:", X_val_ds.shape)
print("Downsampled Test:", X_test_ds.shape)

# ============================================================
# 6. Create PyTorch datasets
# ============================================================

train_dataset = TensorDataset(
    torch.tensor(X_train_ds, dtype=torch.float32),
    torch.tensor(y_train, dtype=torch.float32)
)

val_dataset = TensorDataset(
    torch.tensor(X_val_ds, dtype=torch.float32),
    torch.tensor(y_val, dtype=torch.float32)
)

test_dataset = TensorDataset(
    torch.tensor(X_test_ds, dtype=torch.float32),
    torch.tensor(y_test, dtype=torch.float32)
)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

# ============================================================
# 7. xLSTM-style model
# Note: This is an xLSTM-style recurrent model using LSTM blocks.
# ============================================================

class XLSTMStyleStressClassifier(nn.Module):
    def __init__(self, input_size=1, hidden_size=64, num_layers=2):
        super(XLSTMStyleStressClassifier, self).__init__()

        self.input_projection = nn.Linear(input_size, hidden_size)

        self.lstm = nn.LSTM(
            input_size=hidden_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=0.3
        )

        self.classifier = nn.Sequential(
            nn.Linear(hidden_size, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, 1)
        )

    def forward(self, x):
        x = self.input_projection(x)
        output, _ = self.lstm(x)
        last_output = output[:, -1, :]
        logits = self.classifier(last_output).squeeze(1)
        return logits

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

model = XLSTMStyleStressClassifier().to(device)

criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# ============================================================
# 8. Evaluation function
# ============================================================

def evaluate_model(loader):
    model.eval()

    y_true = []
    y_pred = []
    y_prob = []

    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(device)

            logits = model(xb)
            probs = torch.sigmoid(logits).cpu().numpy()
            preds = (probs >= 0.5).astype(int)

            y_prob.extend(probs)
            y_pred.extend(preds)
            y_true.extend(yb.numpy())

    acc = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred)
    kappa = cohen_kappa_score(y_true, y_pred)

    try:
        auc = roc_auc_score(y_true, y_prob)
    except:
        auc = 0.0

    return acc, f1, kappa, auc

# ============================================================
# 9. Train model
# ============================================================

epochs = 10

for epoch in range(epochs):
    model.train()

    train_true = []
    train_pred = []

    for xb, yb in train_loader:
        xb = xb.to(device)
        yb = yb.to(device)

        optimizer.zero_grad()

        logits = model(xb)
        loss = criterion(logits, yb)

        loss.backward()
        optimizer.step()

        probs = torch.sigmoid(logits).detach().cpu().numpy()
        preds = (probs >= 0.5).astype(int)

        train_pred.extend(preds)
        train_true.extend(yb.cpu().numpy())

    train_acc = accuracy_score(train_true, train_pred)

    val_acc, val_f1, val_kappa, val_auc = evaluate_model(val_loader)

    print(
        f"Epoch {epoch+1}/{epochs} | "
        f"Train Acc: {train_acc:.4f} | "
        f"Val Acc: {val_acc:.4f} | "
        f"Val F1: {val_f1:.4f} | "
        f"Val Kappa: {val_kappa:.4f} | "
        f"Val AUC: {val_auc:.4f}"
    )

# ============================================================
# 10. Final test results
# ============================================================

test_acc, test_f1, test_kappa, test_auc = evaluate_model(test_loader)

print("\n===== Final xLSTM-style Results =====")
print("Training Accuracy:", round(train_acc, 4))
print("Validation Accuracy:", round(val_acc, 4))
print("Test Accuracy:", round(test_acc, 4))
print("F1 Score:", round(test_f1, 4))
print("Kappa:", round(test_kappa, 4))
print("ROC-AUC:", round(test_auc, 4))

import pandas as pd

xlstm_results_df = pd.DataFrame([[
    "xLSTM-style LSTM",
    round(train_acc, 4),
    round(val_acc, 4),
    round(test_acc, 4),
    round(test_f1, 4),
    round(test_kappa, 4),
    round(test_auc, 4)
]], columns=[
    "Model",
    "Train Accuracy",
    "Validation Accuracy",
    "Test Accuracy",
    "F1 Score",
    "Kappa",
    "ROC-AUC"
])

display(xlstm_results_df)

xlstm_results_df.to_csv("xlstm_style_lstm_results.csv", index=False)
print("Saved results as xlstm_style_lstm_results.csv")

Using dataset path: /kaggle/input/datasets/orvile/wesad-wearable-stress-affect-detection-dataset/WESAD
Dataset folders: ['S14', 'S11', 'S13', 'S10', 'S8', 'S5', 'S7', 'S9', 'S15', 'wesad_readme.pdf']
Final X shape: (2749, 7000)
Final y shape: (2749,)
Class distribution: [1762  987]
Train: (1924, 7000, 1)
Validation: (412, 7000, 1)
Test: (413, 7000, 1)
Downsampled Train: (1924, 700, 1)
Downsampled Validation: (412, 700, 1)
Downsampled Test: (413, 700, 1)
Using device: cpu
Epoch 1/10 | Train Acc: 0.6409 | Val Acc: 0.6408 | Val F1: 0.0000 | Val Kappa: 0.0000 | Val AUC: 0.6172
Epoch 2/10 | Train Acc: 0.6409 | Val Acc: 0.6408 | Val F1: 0.0000 | Val Kappa: 0.0000 | Val AUC: 0.6217
Epoch 3/10 | Train Acc: 0.6409 | Val Acc: 0.6408 | Val F1: 0.0000 | Val Kappa: 0.0000 | Val AUC: 0.6633
Epoch 4/10 | Train Acc: 0.6409 | Val Acc: 0.6408 | Val F1: 0.0000 | Val Kappa: 0.0000 | Val AUC: 0.5991
Epoch 5/10 | Train Acc: 0.6409 | Val Acc: 0.6408 | Val F1: 0.0000 | Val Kappa: 0.0000 | Val AUC: 0.6641
Epoc

,Model,Train Accuracy,Validation Accuracy,Test Accuracy,F1 Score,Kappa,ROC-AUC
0,xLSTM-style LSTM,0.6409,0.6408,0.6416,0.0,0.0,0.5959


Saved results as xlstm_style_lstm_results.csv


In [5]:
# ============================================================
# WESAD Stress Detection - Method 2: CDIL-CNN
# Dataset: WESAD ECG, Binary Classification: Baseline vs Stress
# ============================================================

import os
os.environ["CUDA_VISIBLE_DEVICES"] = "-1"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

import pickle
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, f1_score, cohen_kappa_score,
    roc_auc_score, recall_score, precision_score,
    precision_recall_curve, confusion_matrix,
    classification_report
)

import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input, Conv1D, Dense, Dropout, BatchNormalization,
    Activation, GlobalAveragePooling1D, Lambda, Add
)
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.losses import BinaryCrossentropy

# ============================================================
# 1. Auto-detect WESAD dataset path
# ============================================================

possible_paths = []

for root, dirs, files in os.walk("/kaggle/input"):
    if "S2" in dirs:
        possible_paths.append(root)

if len(possible_paths) == 0:
    raise FileNotFoundError(
        "WESAD dataset not found. Click Add Data in Kaggle and attach the WESAD dataset."
    )

BASE_PATH = possible_paths[0]
print("Using BASE_PATH:", BASE_PATH)
print("Dataset folders:", os.listdir(BASE_PATH)[:10])

# ============================================================
# 2. Load and preprocess ECG data
# ============================================================

fs = 700
window_seconds = 10
window_size = fs * window_seconds

all_X = []
all_y = []

for subject in sorted(os.listdir(BASE_PATH)):
    subject_path = os.path.join(BASE_PATH, subject)

    if not os.path.isdir(subject_path):
        continue

    pkl_path = os.path.join(subject_path, f"{subject}.pkl")

    if not os.path.exists(pkl_path):
        continue

    with open(pkl_path, "rb") as f:
        data = pickle.load(f, encoding="latin1")

    ecg = data["signal"]["chest"]["ECG"].reshape(-1)
    labels = data["label"]

    mask = np.isin(labels, [1, 2])
    ecg = ecg[mask]
    labels = labels[mask]

    labels = np.where(labels == 2, 1, 0)

    for start in range(0, len(ecg) - window_size, window_size):
        end = start + window_size

        window = ecg[start:end]
        label_window = labels[start:end]

        stress_ratio = np.mean(label_window)

        if stress_ratio >= 0.5:
            label = 1
        else:
            label = 0

        all_X.append(window)
        all_y.append(label)

X = np.array(all_X)
y = np.array(all_y)

print("Original X shape:", X.shape)
print("Original y shape:", y.shape)
print("Class distribution:", np.bincount(y))

# Normalize each ECG window
X = (X - X.mean(axis=1, keepdims=True)) / (X.std(axis=1, keepdims=True) + 1e-8)

# Downsample from 7000 to 700
X = X[:, ::10]
X = X[..., np.newaxis]

print("Downsampled X shape:", X.shape)

# ============================================================
# 3. Train / Validation / Test Split
# ============================================================

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.50,
    random_state=42,
    stratify=y_temp
)

print("Train:", X_train.shape)
print("Validation:", X_val.shape)
print("Test:", X_test.shape)

# ============================================================
# 4. Class Weight Fix
# ============================================================

class_counts = np.bincount(y_train)
total = len(y_train)

class_weight = {
    0: total / (2 * class_counts[0]),
    1: total / (2 * class_counts[1])
}

print("Class weights:", class_weight)

# ============================================================
# 5. CDIL-CNN Model
# ============================================================

def circular_padding_1d(x, padding):
    left_pad = x[:, -padding:, :]
    right_pad = x[:, :padding, :]
    return tf.concat([left_pad, x, right_pad], axis=1)

def cdil_block(x, filters, kernel_size, dilation_rate, dropout_rate=0.2):
    padding = dilation_rate * (kernel_size - 1) // 2

    x_pad = Lambda(lambda t: circular_padding_1d(t, padding))(x)

    y = Conv1D(
        filters=filters,
        kernel_size=kernel_size,
        dilation_rate=dilation_rate,
        padding="valid"
    )(x_pad)

    y = BatchNormalization()(y)
    y = Activation("relu")(y)
    y = Dropout(dropout_rate)(y)

    if x.shape[-1] != filters:
        x = Conv1D(filters, kernel_size=1, padding="same")(x)

    out = Add()([x, y])
    return out

def build_cdil_cnn(input_shape):
    inputs = Input(shape=input_shape)

    x = Conv1D(64, kernel_size=3, padding="same", activation="relu")(inputs)

    x = cdil_block(x, filters=64, kernel_size=3, dilation_rate=1)
    x = cdil_block(x, filters=64, kernel_size=3, dilation_rate=2)
    x = cdil_block(x, filters=64, kernel_size=3, dilation_rate=4)
    x = cdil_block(x, filters=64, kernel_size=3, dilation_rate=8)
    x = cdil_block(x, filters=64, kernel_size=3, dilation_rate=16)

    x = GlobalAveragePooling1D()(x)
    x = Dense(64, activation="relu")(x)
    x = Dropout(0.3)(x)

    outputs = Dense(1, activation="sigmoid")(x)

    model = Model(inputs, outputs, name="CDIL_CNN")
    return model

model = build_cdil_cnn(input_shape=(X_train.shape[1], X_train.shape[2]))

model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss=BinaryCrossentropy(),
    metrics=["accuracy"]
)

model.summary()

# ============================================================
# 6. Train Model
# ============================================================

print("\nTraining CDIL-CNN...")

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=15,
    batch_size=32,
    class_weight=class_weight,
    callbacks=[
        EarlyStopping(
            monitor="val_loss",
            patience=4,
            restore_best_weights=True
        )
    ],
    verbose=1
)

# ============================================================
# 7. Evaluation with Best Threshold
# ============================================================

train_loss, train_acc = model.evaluate(X_train, y_train, verbose=0)
val_loss, val_acc = model.evaluate(X_val, y_val, verbose=0)

y_prob = model.predict(X_test, verbose=0).reshape(-1)

precision_vals, recall_vals, thresholds = precision_recall_curve(y_test, y_prob)
f1_vals = 2 * (precision_vals * recall_vals) / (precision_vals + recall_vals + 1e-8)

best_index = np.argmax(f1_vals[:-1])
best_threshold = thresholds[best_index]

print("\nBest Threshold:", round(best_threshold, 4))

y_pred = (y_prob >= best_threshold).astype(int)

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

results_df = pd.DataFrame([[
    "CDIL-CNN",
    round(train_acc, 4),
    round(val_acc, 4),
    round(accuracy_score(y_test, y_pred), 4),
    round(f1_score(y_test, y_pred), 4),
    round(cohen_kappa_score(y_test, y_pred), 4),
    round(recall_score(y_test, y_pred), 4),
    round(precision_score(y_test, y_pred), 4),
    round(roc_auc_score(y_test, y_prob), 4),
    round(best_threshold, 4)
]], columns=[
    "Model",
    "Train Accuracy",
    "Validation Accuracy",
    "Test Accuracy",
    "F1 Score",
    "Kappa",
    "Recall",
    "Precision",
    "ROC-AUC",
    "Best Threshold"
])
print("\n================ CDIL-CNN RESULTS ================")
display(results_df)

results_df.to_csv("cdil_cnn_results.csv", index=False)
print("Saved results as cdil_cnn_results.csv")

Using BASE_PATH: /kaggle/input/datasets/orvile/wesad-wearable-stress-affect-detection-dataset/WESAD
Dataset folders: ['S14', 'S11', 'S13', 'S10', 'S8', 'S5', 'S7', 'S9', 'S15', 'wesad_readme.pdf']
Original X shape: (2749, 7000)
Original y shape: (2749,)
Class distribution: [1761  988]
Downsampled X shape: (2749, 700, 1)
Train: (1924, 700, 1)
Validation: (412, 700, 1)
Test: (413, 700, 1)
Class weights: {0: np.float64(0.7802108678021087), 1: np.float64(1.3921852387843705)}


Model: "CDIL_CNN"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_4       │ (None, 700, 1)    │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_3 (Conv1D)   │ (None, 700, 64)   │        256 │ input_layer_4[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lambda (Lambda)     │ (None, 702, 64)   │          0 │ conv1d_3[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_4 (Conv1D)   │ (None, 700, 64)   │     12,352 │ lambda[0][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalization │ (None, 700, 64)   │        256 │ conv1d_4[0][0]    │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation          │ (None, 700, 64)   │          0 │ batch_normalizat… │
│ (Activation)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_5 (Dropout) │ (None, 700, 64)   │          0 │ activation[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add (Add)           │ (None, 700, 64)   │          0 │ conv1d_3[0][0],   │
│                     │                   │            │ dropout_5[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lambda_1 (Lambda)   │ (None, 704, 64)   │          0 │ add[0][0]         │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_5 (Conv1D)   │ (None, 700, 64)   │     12,352 │ lambda_1[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 700, 64)   │        256 │ conv1d_5[0][0]    │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_1        │ (None, 700, 64)   │          0 │ batch_normalizat… │
│ (Activation)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_6 (Dropout) │ (None, 700, 64)   │          0 │ activation_1[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_1 (Add)         │ (None, 700, 64)   │          0 │ add[0][0],        │
│                     │                   │            │ dropout_6[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lambda_2 (Lambda)   │ (None, 708, 64)   │          0 │ add_1[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_6 (Conv1D)   │ (None, 700, 64)   │     12,352 │ lambda_2[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 700, 64)   │        256 │ conv1d_6[0][0]    │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_2        │ (None, 700, 64)   │          0 │ batch_normalizat… │
│ (Activation)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_7 (Dropout) │ (None, 700, 64)   │          0 │ activation_2[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_2 (Add)         │ (None, 700, 64)   │          0 │ add_1[0][0],      │
│                     │                   │            │ dropout_7[0][0] 

 Total params: 67,521 (263.75 KB)

 Trainable params: 66,881 (261.25 KB)

 Non-trainable params: 640 (2.50 KB)


Training CDIL-CNN...
Epoch 1/15
61/61 ━━━━━━━━━━━━━━━━━━━━ 24s 275ms/step - accuracy: 0.6616 - loss: 0.6945 - val_accuracy: 0.6408 - val_loss: 0.6494
Epoch 2/15
61/61 ━━━━━━━━━━━━━━━━━━━━ 16s 256ms/step - accuracy: 0.8112 - loss: 0.3947 - val_accuracy: 0.6408 - val_loss: 0.6936
Epoch 3/15
61/61 ━━━━━━━━━━━━━━━━━━━━ 17s 272ms/step - accuracy: 0.8808 - loss: 0.2725 - val_accuracy: 0.7743 - val_loss: 0.4870
Epoch 4/15
61/61 ━━━━━━━━━━━━━━━━━━━━ 16s 263ms/step - accuracy: 0.8926 - loss: 0.2471 - val_accuracy: 0.4005 - val_loss: 1.5952
Epoch 5/15
61/61 ━━━━━━━━━━━━━━━━━━━━ 16s 270ms/step - accuracy: 0.9198 - loss: 0.1922 - val_accuracy: 0.9150 - val_loss: 0.1993
Epoch 6/15
61/61 ━━━━━━━━━━━━━━━━━━━━ 16s 263ms/step - accuracy: 0.9409 - loss: 0.1706 - val_accuracy: 0.9102 - val_loss: 0.1693
Epoch 7/15
61/61 ━━━━━━━━━━━━━━━━━━━━ 16s 262ms/step - accuracy: 0.9179 - loss: 0.1718 - val_accuracy: 0.8641 - val_loss: 0.2914
Epoch 8/15
61/61 ━━━━━━━━━━━━━━━━━━━━ 16s 263ms/step - accuracy: 0.9540 - l

,Model,Train Accuracy,Validation Accuracy,Test Accuracy,F1 Score,Kappa,Recall,Precision,ROC-AUC,Best Threshold
0,CDIL-CNN,0.9673,0.9612,0.9855,0.9799,0.9685,0.9799,0.9799,0.9968,0.0851


Saved results as cdil_cnn_results.csv


In [6]:
# ============================================================
# WESAD Stress Detection - Method 3: CNN-BiLSTM-Attention
# Dataset: WESAD ECG, Binary Classification: Baseline vs Stress
# ============================================================

import os
os.environ["CUDA_VISIBLE_DEVICES"] = "-1"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

import pickle
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, f1_score, cohen_kappa_score,
    roc_auc_score, recall_score, precision_score,
    precision_recall_curve, confusion_matrix,
    classification_report
)

import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input, Conv1D, Dense, Dropout, MaxPooling1D,
    Bidirectional, LSTM, Multiply, Lambda, Softmax
)
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.losses import BinaryCrossentropy

# ============================================================
# 1. Auto-detect WESAD dataset path
# ============================================================

possible_paths = []

for root, dirs, files in os.walk("/kaggle/input"):
    if "S2" in dirs:
        possible_paths.append(root)

if len(possible_paths) == 0:
    raise FileNotFoundError(
        "WESAD dataset not found. Click Add Data in Kaggle and attach the WESAD dataset."
    )

BASE_PATH = possible_paths[0]
print("Using BASE_PATH:", BASE_PATH)
print("Dataset folders:", os.listdir(BASE_PATH)[:10])

# ============================================================
# 2. Load and preprocess ECG data
# ============================================================

fs = 700
window_seconds = 10
window_size = fs * window_seconds

all_X = []
all_y = []

for subject in sorted(os.listdir(BASE_PATH)):
    subject_path = os.path.join(BASE_PATH, subject)

    if not os.path.isdir(subject_path):
        continue

    pkl_path = os.path.join(subject_path, f"{subject}.pkl")

    if not os.path.exists(pkl_path):
        continue

    with open(pkl_path, "rb") as f:
        data = pickle.load(f, encoding="latin1")

    ecg = data["signal"]["chest"]["ECG"].reshape(-1)
    labels = data["label"]

    # Keep only baseline = 1 and stress = 2
    mask = np.isin(labels, [1, 2])
    ecg = ecg[mask]
    labels = labels[mask]

    # baseline = 0, stress = 1
    labels = np.where(labels == 2, 1, 0)

    for start in range(0, len(ecg) - window_size, window_size):
        end = start + window_size

        window = ecg[start:end]
        label_window = labels[start:end]

        stress_ratio = np.mean(label_window)

        if stress_ratio >= 0.5:
            label = 1
        else:
            label = 0

        all_X.append(window)
        all_y.append(label)

X = np.array(all_X)
y = np.array(all_y)

print("Original X shape:", X.shape)
print("Original y shape:", y.shape)
print("Class distribution:", np.bincount(y))

# Normalize each ECG window
X = (X - X.mean(axis=1, keepdims=True)) / (X.std(axis=1, keepdims=True) + 1e-8)

# Downsample ECG from 7000 to 700 for faster training
X = X[:, ::10]
X = X[..., np.newaxis]

print("Downsampled X shape:", X.shape)

# ============================================================
# 3. Train / Validation / Test Split
# ============================================================

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.50,
    random_state=42,
    stratify=y_temp
)

print("Train:", X_train.shape)
print("Validation:", X_val.shape)
print("Test:", X_test.shape)

# ============================================================
# 4. Class Weight Fix
# ============================================================

class_counts = np.bincount(y_train)
total = len(y_train)

class_weight = {
    0: total / (2 * class_counts[0]),
    1: total / (2 * class_counts[1])
}

print("Class weights:", class_weight)

# ============================================================
# 5. CNN-BiLSTM-Attention Model
# ============================================================

def attention_layer(inputs):
    score = Dense(1, activation="tanh")(inputs)
    weights = Softmax(axis=1)(score)
    context = Multiply()([inputs, weights])
    context = Lambda(lambda x: tf.reduce_sum(x, axis=1))(context)
    return context

def build_cnn_bilstm_attention(input_shape):
    inputs = Input(shape=input_shape)

    x = Conv1D(64, kernel_size=5, padding="same", activation="relu")(inputs)
    x = MaxPooling1D(pool_size=2)(x)
    x = Dropout(0.2)(x)

    x = Conv1D(128, kernel_size=3, padding="same", activation="relu")(x)
    x = MaxPooling1D(pool_size=2)(x)
    x = Dropout(0.2)(x)

    x = Bidirectional(LSTM(64, return_sequences=True))(x)

    x = attention_layer(x)

    x = Dense(64, activation="relu")(x)
    x = Dropout(0.3)(x)

    outputs = Dense(1, activation="sigmoid")(x)

    model = Model(inputs, outputs, name="CNN_BiLSTM_Attention")
    return model

model = build_cnn_bilstm_attention(
    input_shape=(X_train.shape[1], X_train.shape[2])
)

model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss=BinaryCrossentropy(),
    metrics=["accuracy"]
)

model.summary()

# ============================================================
# 6. Train Model
# ============================================================

print("\nTraining CNN-BiLSTM-Attention...")

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=15,
    batch_size=32,
    class_weight=class_weight,
    callbacks=[
        EarlyStopping(
            monitor="val_loss",
            patience=4,
            restore_best_weights=True
        )
    ],
    verbose=1
)

# ============================================================
# 7. Evaluation with Best Threshold
# ============================================================

train_loss, train_acc = model.evaluate(X_train, y_train, verbose=0)
val_loss, val_acc = model.evaluate(X_val, y_val, verbose=0)

y_prob = model.predict(X_test, verbose=0).reshape(-1)

precision_vals, recall_vals, thresholds = precision_recall_curve(y_test, y_prob)
f1_vals = 2 * (precision_vals * recall_vals) / (precision_vals + recall_vals + 1e-8)

best_index = np.argmax(f1_vals[:-1])
best_threshold = thresholds[best_index]

print("\nBest Threshold:", round(best_threshold, 4))

y_pred = (y_prob >= best_threshold).astype(int)

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

results_df = pd.DataFrame([[
    "CNN-BiLSTM-Attention",
    round(train_acc, 4),
    round(val_acc, 4),
    round(accuracy_score(y_test, y_pred), 4),
    round(f1_score(y_test, y_pred), 4),
    round(cohen_kappa_score(y_test, y_pred), 4),
    round(recall_score(y_test, y_pred), 4),
    round(precision_score(y_test, y_pred), 4),
    round(roc_auc_score(y_test, y_prob), 4),
    round(best_threshold, 4)
]], columns=[
    "Model",
    "Train Accuracy",
    "Validation Accuracy",
    "Test Accuracy",
    "F1 Score",
    "Kappa",
    "Recall",
    "Precision",
    "ROC-AUC",
    "Best Threshold"
])
print("\n================ CNN-BiLSTM-ATTENTION RESULTS ================")
display(results_df)

results_df.to_csv("cnn_bilstm_attention_results.csv", index=False)
print("Saved results as cnn_bilstm_attention_results.csv")

Using BASE_PATH: /kaggle/input/datasets/orvile/wesad-wearable-stress-affect-detection-dataset/WESAD
Dataset folders: ['S14', 'S11', 'S13', 'S10', 'S8', 'S5', 'S7', 'S9', 'S15', 'wesad_readme.pdf']
Original X shape: (2749, 7000)
Original y shape: (2749,)
Class distribution: [1761  988]
Downsampled X shape: (2749, 700, 1)
Train: (1924, 700, 1)
Validation: (412, 700, 1)
Test: (413, 700, 1)
Class weights: {0: np.float64(0.7802108678021087), 1: np.float64(1.3921852387843705)}


Model: "CNN_BiLSTM_Attention"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_5       │ (None, 700, 1)    │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_9 (Conv1D)   │ (None, 700, 64)   │        384 │ input_layer_5[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling1d_3     │ (None, 350, 64)   │          0 │ conv1d_9[0][0]    │
│ (MaxPooling1D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_11          │ (None, 350, 64)   │          0 │ max_pooling1d_3[… │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_10 (Conv1D)  │ (None, 350, 128)  │     24,704 │ dropout_11[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling1d_4     │ (None, 175, 128)  │          0 │ conv1d_10[0][0]   │
│ (MaxPooling1D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_12          │ (None, 175, 128)  │          0 │ max_pooling1d_4[… │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bidirectional_1     │ (None, 175, 128)  │     98,816 │ dropout_12[0][0]  │
│ (Bidirectional)     │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_11 (Dense)    │ (None, 175, 1)    │        129 │ bidirectional_1[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ softmax (Softmax)   │ (None, 175, 1)    │          0 │ dense_11[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multiply (Multiply) │ (None, 175, 128)  │          0 │ bidirectional_1[… │
│                     │                   │            │ softmax[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lambda_5 (Lambda)   │ (None, 128)       │          0 │ multiply[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_12 (Dense)    │ (None, 64)        │      8,256 │ lambda_5[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_13          │ (None, 64)        │          0 │ dense_12[0][0]    │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_13 (Dense)    │ (None, 1)         │         65 │ dropout_13[0][0]  │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 132,354 (517.01 KB)

 Trainable params: 132,354 (517.01 KB)

 Non-trainable params: 0 (0.00 B)


Training CNN-BiLSTM-Attention...
Epoch 1/15
61/61 ━━━━━━━━━━━━━━━━━━━━ 16s 182ms/step - accuracy: 0.4554 - loss: 0.6914 - val_accuracy: 0.7524 - val_loss: 0.6033
Epoch 2/15
61/61 ━━━━━━━━━━━━━━━━━━━━ 10s 171ms/step - accuracy: 0.6937 - loss: 0.6104 - val_accuracy: 0.8107 - val_loss: 0.4860
Epoch 3/15
61/61 ━━━━━━━━━━━━━━━━━━━━ 11s 188ms/step - accuracy: 0.7410 - loss: 0.5217 - val_accuracy: 0.8374 - val_loss: 0.3956
Epoch 4/15
61/61 ━━━━━━━━━━━━━━━━━━━━ 13s 208ms/step - accuracy: 0.7719 - loss: 0.4399 - val_accuracy: 0.8568 - val_loss: 0.3116
Epoch 5/15
61/61 ━━━━━━━━━━━━━━━━━━━━ 15s 249ms/step - accuracy: 0.8157 - loss: 0.3840 - val_accuracy: 0.8617 - val_loss: 0.3295
Epoch 6/15
61/61 ━━━━━━━━━━━━━━━━━━━━ 12s 195ms/step - accuracy: 0.8160 - loss: 0.3789 - val_accuracy: 0.9102 - val_loss: 0.2110
Epoch 7/15
61/61 ━━━━━━━━━━━━━━━━━━━━ 13s 205ms/step - accuracy: 0.8742 - loss: 0.2682 - val_accuracy: 0.9223 - val_loss: 0.1855
Epoch 8/15
61/61 ━━━━━━━━━━━━━━━━━━━━ 12s 193ms/step - accuracy

,Model,Train Accuracy,Validation Accuracy,Test Accuracy,F1 Score,Kappa,Recall,Precision,ROC-AUC,Best Threshold
0,CNN-BiLSTM-Attention,0.9667,0.9563,0.9588,0.9424,0.9104,0.9329,0.9521,0.9911,0.465


Saved results as cnn_bilstm_attention_results.csv
